# Funnel and Conversion Analysis

## Overview
A **funnel** models the sequential steps users take toward a goal. At each step some users drop off, so the population narrows -- like a funnel. Conversion analysis measures how many users reach each step and identifies where the biggest losses occur.

**Typical account-opening funnel:**
```
page_view -> product_view -> application_start -> application_submit -> account_opened
```

**Key metrics per step:**

| Metric | Formula |
|---|---|
| Step volume | COUNT(DISTINCT users at this step) |
| Step conversion rate | users at step N / users at step N-1 |
| Overall conversion rate | users at final step / users at first step |
| Drop-off rate | 1 - step conversion rate |

**Funnel types:**
- **Strict / ordered funnel** -- user must complete steps in sequence; earlier steps are prerequisites
- **Unordered funnel** -- steps can happen in any order; we just check whether each happened
- **Time-boxed funnel** -- user must complete the entire funnel within a defined window (e.g. 7 days)

**PostgreSQL notes:** Window functions and CTEs are the standard tools. `LEAD()` and `LAG()` help detect whether a user advanced to the next step. `FILTER (WHERE ...)` inside aggregates is a clean PostgreSQL alternative to `CASE WHEN ... END` inside `COUNT`.

---

In [ ]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
conn.executescript("""CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, first_name TEXT, last_name TEXT, segment TEXT, province TEXT, signup_date TEXT, active INTEGER DEFAULT 1);CREATE TABLE accounts (account_id INTEGER PRIMARY KEY, customer_id INTEGER, account_type TEXT, opened_date TEXT, status TEXT, credit_limit REAL);CREATE TABLE transactions (txn_id INTEGER PRIMARY KEY, account_id INTEGER, customer_id INTEGER, txn_date TEXT, txn_type TEXT, amount REAL, category TEXT);CREATE TABLE product_events (event_id INTEGER PRIMARY KEY, customer_id INTEGER, event_date TEXT, event_type TEXT, product TEXT, channel TEXT);INSERT INTO customers VALUES (1,'Aroha','Ngata','Retail','NB','2023-01-05',1),(2,'Liam','Chen','Wealth','NS','2023-01-12',1),(3,'Fatima','Al-Rashid','SME','ON','2023-01-20',1),(4,'James','MacLeod','Retail','NB','2023-02-03',1),(5,'Sofia','Petrov','Retail','BC','2023-02-14',1),(6,'Noah','Williams','SME','AB','2023-02-28',0),(7,'Mei','Zhang','Wealth','ON','2023-03-10',1),(8,'Ethan','Tremblay','Retail','QC','2023-04-05',1),(9,'Priya','Nair','Wealth','BC','2023-04-18',1),(10,'Marcus','Okafor','SME','ON','2023-04-25',1),(11,'Diana','Flores','Retail','NB','2023-05-02',1),(12,'Ravi','Patel','Retail','ON','2023-05-15',1),(13,'Yuki','Tanaka','Wealth','BC','2023-06-01',1),(14,'Omar','Hassan','SME','QC','2023-06-20',1),(15,'Chloe','Bouchard','Retail','QC','2023-07-08',0);INSERT INTO accounts VALUES (101,1,'Chequing','2023-01-05','Active',NULL),(102,1,'Savings','2023-01-05','Active',NULL),(103,2,'Investment','2023-01-12','Active',50000),(104,3,'Chequing','2023-01-20','Active',NULL),(105,3,'Loan','2023-01-20','Active',25000),(106,4,'Chequing','2023-02-03','Active',NULL),(107,5,'Chequing','2023-02-14','Active',NULL),(108,5,'Savings','2023-02-14','Active',NULL),(109,6,'Chequing','2023-02-28','Closed',NULL),(110,7,'Investment','2023-03-10','Active',75000),(111,8,'Chequing','2023-04-05','Active',NULL),(112,9,'Investment','2023-04-18','Active',100000),(113,10,'Chequing','2023-04-25','Active',NULL),(114,10,'Loan','2023-04-25','Active',15000),(115,11,'Chequing','2023-05-02','Active',NULL),(116,12,'Chequing','2023-05-15','Active',NULL),(117,12,'Savings','2023-05-15','Active',NULL),(118,13,'Investment','2023-06-01','Active',60000),(119,14,'Chequing','2023-06-20','Active',NULL),(120,15,'Chequing','2023-07-08','Closed',NULL);INSERT INTO transactions VALUES (1001,101,1,'2023-01-10','Deposit',2500.00,'Payroll'),(1002,101,1,'2023-01-22','Withdrawal',-800.00,'Rent'),(1003,102,1,'2023-02-05','Deposit',500.00,'Transfer'),(1004,103,2,'2023-02-10','Deposit',10000.00,'Investment'),(1005,104,3,'2023-02-15','Deposit',3200.00,'Payroll'),(1006,104,3,'2023-03-01','Withdrawal',-1200.00,'Rent'),(1007,106,4,'2023-03-10','Deposit',2800.00,'Payroll'),(1008,107,5,'2023-03-15','Deposit',2200.00,'Payroll'),(1009,107,5,'2023-03-28','Fee',-25.00,'Monthly Fee'),(1010,101,1,'2023-03-10','Deposit',2500.00,'Payroll'),(1011,103,2,'2023-04-15','Deposit',15000.00,'Investment'),(1012,110,7,'2023-04-20','Deposit',20000.00,'Investment'),(1013,111,8,'2023-05-01','Deposit',2100.00,'Payroll'),(1014,112,9,'2023-05-10','Deposit',25000.00,'Investment'),(1015,113,10,'2023-05-20','Deposit',3500.00,'Payroll'),(1016,115,11,'2023-06-01','Deposit',2000.00,'Payroll'),(1017,116,12,'2023-06-15','Deposit',2700.00,'Payroll'),(1018,118,13,'2023-07-01','Deposit',18000.00,'Investment'),(1019,101,1,'2023-07-10','Deposit',2500.00,'Payroll'),(1020,103,2,'2023-07-20','Deposit',12000.00,'Investment'),(1021,104,3,'2023-07-15','Deposit',3200.00,'Payroll'),(1022,107,5,'2023-08-01','Deposit',2200.00,'Payroll'),(1023,111,8,'2023-08-05','Deposit',2100.00,'Payroll'),(1024,113,10,'2023-08-15','Withdrawal',-500.00,'Transfer'),(1025,116,12,'2023-08-20','Deposit',2700.00,'Payroll'),(1026,101,1,'2023-10-10','Deposit',2500.00,'Payroll'),(1027,116,12,'2023-10-20','Deposit',2700.00,'Payroll');INSERT INTO product_events VALUES (1,1,'2023-01-10','page_view','Savings Account','web'),(2,1,'2023-01-10','product_view','Savings Account','web'),(3,1,'2023-01-10','application_start','Savings Account','web'),(4,1,'2023-01-11','application_submit','Savings Account','web'),(5,1,'2023-01-15','account_opened','Savings Account','web'),(6,2,'2023-01-12','page_view','Investment Account','mobile'),(7,2,'2023-01-12','product_view','Investment Account','mobile'),(8,2,'2023-01-12','application_start','Investment Account','mobile'),(9,2,'2023-01-13','application_submit','Investment Account','mobile'),(10,2,'2023-01-15','account_opened','Investment Account','mobile'),(11,3,'2023-01-20','page_view','Chequing Account','web'),(12,3,'2023-01-20','product_view','Chequing Account','web'),(13,3,'2023-01-20','application_start','Chequing Account','web'),(14,4,'2023-02-03','page_view','Chequing Account','web'),(15,4,'2023-02-03','product_view','Chequing Account','web'),(16,5,'2023-02-14','page_view','Savings Account','mobile'),(17,5,'2023-02-14','product_view','Savings Account','mobile'),(18,5,'2023-02-14','application_start','Savings Account','mobile'),(19,5,'2023-02-15','application_submit','Savings Account','mobile'),(20,5,'2023-02-20','account_opened','Savings Account','mobile'),(21,7,'2023-03-10','page_view','Investment Account','web'),(22,7,'2023-03-10','product_view','Investment Account','web'),(23,7,'2023-03-10','application_start','Investment Account','web'),(24,7,'2023-03-11','application_submit','Investment Account','web'),(25,7,'2023-03-15','account_opened','Investment Account','web'),(26,8,'2023-04-05','page_view','Chequing Account','mobile'),(27,8,'2023-04-05','product_view','Chequing Account','mobile'),(28,9,'2023-04-18','page_view','Investment Account','web'),(29,9,'2023-04-18','product_view','Investment Account','web'),(30,9,'2023-04-18','application_start','Investment Account','web'),(31,9,'2023-04-20','application_submit','Investment Account','web'),(32,9,'2023-04-25','account_opened','Investment Account','web');""")
conn.commit()
print("Finance schema ready.")
for t in ["customers","accounts","transactions","product_events"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t}: {n} rows")

---
## Step volumes and conversion rates

In [ ]:
print("=== Funnel step volumes ===")
q = """
WITH funnel_steps AS (
    SELECT  event_type,
            COUNT(DISTINCT customer_id)  AS users
    FROM    product_events
    WHERE   event_type IN ('page_view','product_view','application_start',
                           'application_submit','account_opened')
    GROUP BY event_type
),
ordered AS (
    SELECT  event_type,
            users,
            CASE event_type
                WHEN 'page_view'          THEN 1
                WHEN 'product_view'       THEN 2
                WHEN 'application_start'  THEN 3
                WHEN 'application_submit' THEN 4
                WHEN 'account_opened'     THEN 5
            END AS step_num
    FROM funnel_steps
)
SELECT  step_num,
        event_type                                         AS step,
        users,
        LAG(users) OVER (ORDER BY step_num)                AS prev_users,
        CASE WHEN LAG(users) OVER (ORDER BY step_num) IS NULL THEN NULL
             ELSE ROUND(100.0 * users /
                        LAG(users) OVER (ORDER BY step_num), 1)
        END                                                AS step_conv_pct,
        ROUND(100.0 * users /
              FIRST_VALUE(users) OVER (ORDER BY step_num), 1) AS overall_conv_pct
FROM    ordered
ORDER BY step_num
"""
print(pd.read_sql(q, conn).to_string(index=False))


---
## Drop-off analysis by channel

In [ ]:
print("=== Funnel by channel: web vs mobile ===")
q = """
WITH step_channel AS (
    SELECT  channel,
            COUNT(DISTINCT CASE WHEN event_type='page_view'          THEN customer_id END) AS s1_page,
            COUNT(DISTINCT CASE WHEN event_type='product_view'       THEN customer_id END) AS s2_product,
            COUNT(DISTINCT CASE WHEN event_type='application_start'  THEN customer_id END) AS s3_start,
            COUNT(DISTINCT CASE WHEN event_type='application_submit' THEN customer_id END) AS s4_submit,
            COUNT(DISTINCT CASE WHEN event_type='account_opened'     THEN customer_id END) AS s5_opened
    FROM    product_events
    GROUP BY channel
)
SELECT  channel,
        s1_page,
        s2_product,
        s3_start,
        s4_submit,
        s5_opened,
        ROUND(100.0 * s5_opened / NULLIF(s1_page,0), 1) AS overall_conv_pct
FROM    step_channel
ORDER BY overall_conv_pct DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))

print()
print("=== Customers who dropped at application_start (never submitted) ===")
q2 = """
SELECT  pe.customer_id,
        c.first_name || ' ' || c.last_name  AS customer,
        c.segment,
        pe.channel,
        pe.product
FROM    product_events AS pe
JOIN    customers      AS c  ON pe.customer_id = c.customer_id
WHERE   pe.event_type = 'application_start'
  AND   pe.customer_id NOT IN (
            SELECT customer_id FROM product_events
            WHERE  event_type = 'application_submit'
        )
"""
print(pd.read_sql(q2, conn).to_string(index=False))


---
## Time-to-convert analysis

In [ ]:
print("=== Days from first page_view to account_opened ===")
q = """
WITH first_view AS (
    SELECT  customer_id, MIN(event_date)  AS first_view_date
    FROM    product_events
    WHERE   event_type = 'page_view'
    GROUP BY customer_id
),
opened AS (
    SELECT  customer_id, MIN(event_date)  AS opened_date
    FROM    product_events
    WHERE   event_type = 'account_opened'
    GROUP BY customer_id
)
SELECT  o.customer_id,
        c.first_name || ' ' || c.last_name  AS customer,
        c.segment,
        fv.first_view_date,
        o.opened_date,
        CAST(JULIANDAY(o.opened_date) - JULIANDAY(fv.first_view_date) AS INTEGER)  AS days_to_convert
FROM    opened     AS o
JOIN    first_view AS fv ON o.customer_id = fv.customer_id
JOIN    customers  AS c  ON o.customer_id = c.customer_id
ORDER BY days_to_convert
"""
df = pd.read_sql(q, conn)
print(df.to_string(index=False))
print()
print(f"Average days to convert: {df['days_to_convert'].mean():.1f}")
print(f"Median days to convert:  {df['days_to_convert'].median():.1f}")
conn.close()


---
## Common Pitfalls

**1. Counting events instead of unique users per step**
If a user views the page 3 times, `COUNT(event_id)` gives 3 but there is still only 1 user in the funnel at that step. Always use `COUNT(DISTINCT customer_id)` per step. Event counts inflate early-stage numbers and make conversion rates look worse than they are.

**2. Not enforcing step ordering in a strict funnel**
A user who submitted an application before viewing the product page (e.g. direct link) will show in step 4 but not step 2. In a strict funnel, users should only count for step N if they completed all prior steps. Use a CTE chain or `MIN(event_date)` ordering to enforce sequence.

**3. Ignoring the time window**
A user who viewed the page in January and opened an account in December looks like a conversion -- but were they really part of the same funnel journey? Define a maximum conversion window (e.g. 30 days) and exclude users who exceeded it. Without a window, time-to-convert statistics are meaningless.

**4. NULLIF omitted in the conversion rate denominator**
`100.0 * s5_opened / s1_page` crashes with division-by-zero if a channel had zero page views. Always wrap the denominator in `NULLIF(col, 0)` to return NULL rather than an error for empty segments.

**5. Funnel with non-exclusive steps**
If a user starts two applications (one for savings, one for chequing), they appear twice at `application_start`. Decide whether the funnel is per-customer or per-product-application and filter accordingly before computing step counts.

**6. Using overall conversion as the only KPI**
Overall conversion (page view to account opened) hides where the problem is. A 5% overall rate with a 90% drop at `application_start` needs a different fix than a 5% rate with a 90% drop at `application_submit`. Always decompose step-by-step.


---
*sql_methods_library - Samantha McGarrigle*